In [1]:

import tqdm as notebook_tqdm
from darts import TimeSeries, datasets
from sktime.forecasting.chronos import ChronosForecaster
from sktime.forecasting.base import ForecastingHorizon
import warnings
from darts.utils.missing_values import fill_missing_values
from sktime.performance_metrics.forecasting import mean_absolute_error, MeanAbsoluteScaledError, mean_absolute_percentage_error


c:\Users\Diat\anaconda3\envs\chrono\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
# Note: Dart import dataset as TimeSeries data, which need to be converted before giving it into the 
#         Forecaster of ant kind, dataFrame is suppourted hence we change Timeseries to DF before giving it to 
#             any function bt SKTime

In [42]:
# warnings.filterwarnings('ignore')
# forecaster = ChronosForecaster("amazon/chronos-t5-small", config={"device_map":"cuda"}) ####cuda

In [43]:
# #### tempreture dataset

# y_nan = datasets.TemperatureDataset().load()
# y = fill_missing_values(y_nan,fill="auto")    ###### HANDLE MISSING VALUES  

# y_train, y_test = y.split_before(len(y)-30)
# fh = ForecastingHorizon(y_test.time_index, is_relative=False)


# forecaster.fit(y_train.to_dataframe())
# y_pred = forecaster.predict(fh)

In [44]:
# mean_absolute_error(y_test.to_dataframe(),y_pred)

In [2]:
import pickle

with open("darts_univariate_catalog_ordered.pkl", "rb") as f:
    data = pickle.load(f)

data

,dataset_name,load_variable,seasonal,split,test_prediction_split,has_missing_values,length,freq
0,AusBeer,((( Y\ndate \n1956-...,True,207.0,4,False,211,<QuarterBegin: startingMonth=10>
1,Wooly,((( Y\ndate \n196...,True,115.0,4,False,119,<QuarterBegin: startingMonth=10>
2,MonthlyMilk,((( Pounds per cow\nMonth ...,True,156.0,12,False,168,<MonthBegin>
3,AirPassengers,((( #Passengers\nMonth ...,True,132.0,12,False,144,<MonthBegin>
4,Sunspots,((( Sunspots\nMonth \...,True,2808.0,12,False,2820,<MonthBegin>
5,MonthlyMilkIncomplete,((( Pounds per cow\nMonth ...,True,156.0,12,True,168,<MonthBegin>
6,Wine,((( Y\ndate \n1...,True,164.0,12,False,176,<MonthBegin>
7,ETTh2_OT,((( OT\ndate ...,True,17396.0,24,False,17420,<Hour>
8,ETTh1_OT,((( OT\ndate \n201...,True,17396.0,24,False,17420,<Hour>
9,TaxiNewYork,((( #Passengers\ntime ...,True,10272.0,48,False,10320,<30 * Minutes>


In [3]:
warnings.filterwarnings('ignore')
forecaster = ChronosForecaster("amazon/chronos-t5-base", config={"device_map":"cuda"}) ####cuda

In [5]:
import os
import matplotlib.pyplot as plt


def plot_forecast(y_train, y_test, y_pred, dataset_name, save_dir="chronos"):

    # Create folder
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(12, 5))

    # Training data
    plt.plot(
        y_train.time_index,
        y_train.values().flatten(),
        label="Train",
        linewidth=1.5
    )

    # Actual test data
    plt.plot(
        y_test.time_index,
        y_test.values().flatten(),
        label="Test",
        linewidth=2
    )

    # Chronos prediction
    plt.plot(
        y_pred.index,
        y_pred.iloc[:, 0].values,
        label="Chronos Prediction",
        linewidth=2,
        linestyle="--"
    )

    plt.title(f"Chronos Forecast - {dataset_name}")
    plt.xlabel("Time")
    plt.ylabel("Value")

    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Make filename Windows-safe
    safe_name = "".join(
        c if c.isalnum() or c in (" ", "_", "-") else "_"
        for c in str(dataset_name)
    ).strip()

    save_path = os.path.join(
        save_dir,
        f"{safe_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    return save_path

In [6]:
results_chronos = []
Mase_con = MeanAbsoluteScaledError()

In [21]:
for i in range(0,16):
    
    y = data["load_variable"][i]
    
    if data["has_missing_values"][i] == True:
        y = fill_missing_values(y,fill="auto")    ###### HANDLE MISSING VALUES

    split_point = int(data["split"][i]) if data["seasonal"][i] else data["split"][i]
    y_train, y_test = y.split_before(split_point)

    MAX_CONTEXT = 2048
    if len(y_train) > MAX_CONTEXT:
        y_train = y_train[-MAX_CONTEXT:]

    
    fh = ForecastingHorizon(y_test.time_index, is_relative=False)

    forecaster.fit(y_train.to_dataframe())
    y_pred = forecaster.predict(fh)


    dataset_name = data["dataset_name"][i]

    plot_forecast(
        y_train=y_train,
        y_test=y_test,
        y_pred=y_pred,
        dataset_name=dataset_name,
        save_dir="chronos"
    )


    mae = mean_absolute_error(y_test.to_dataframe(),y_pred)
    mase = Mase_con(y_test.to_dataframe(), y_pred, y_train=y_train.to_dataframe())
    smape = mean_absolute_percentage_error(y_test.to_dataframe(),y_pred, symmetric = True)

    new_row = {"data": data["dataset_name"][i],"MAE":mae,"mase":mase,"sMAPE":smape}
    results_chronos.append(new_row)



In [22]:
results_chronos

[{'data': 'AusBeer',
  'MAE': np.float64(7.920310974121094),
  'mase': np.float64(0.13869296673486445),
  'sMAPE': np.float64(0.017847907350895247)},
 {'data': 'Wooly',
  'MAE': np.float64(844.450439453125),
  'mase': np.float64(1.5034960736175211),
  'sMAPE': np.float64(0.1588657842888184)},
 {'data': 'MonthlyMilk',
  'MAE': np.float64(7.3999582926432295),
  'mase': np.float64(0.19046720945860188),
  'sMAPE': np.float64(0.008776430465105399)},
 {'data': 'AirPassengers',
  'MAE': np.float64(11.981592814127604),
  'mase': np.float64(0.49749244331242976),
  'sMAPE': np.float64(0.026142366139901955)},
 {'data': 'Sunspots',
  'MAE': np.float64(28.409637196858725),
  'mase': np.float64(2.220435548624138),
  'sMAPE': np.float64(0.39506802665387336)},
 {'data': 'MonthlyMilkIncomplete',
  'MAE': np.float64(11.777074178059896),
  'mase': np.float64(0.31013362174639547),
  'sMAPE': np.float64(0.014023991179076071)},
 {'data': 'Wine',
  'MAE': np.float64(2005.7249348958333),
  'mase': np.float64(

In [23]:
import pandas as pd

In [24]:
chronos_results_df = pd.DataFrame(results_chronos)

In [25]:
chronos_results_df.to_pickle("chronos_results_base_3.pkl")